In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.io import loadmat
from scipy.stats import kurtosis, skew

In [3]:
dataset_path = Path(
    r"C:\Users\suthi\Downloads\AI_Motor_Predictive_Maintenance\datasets"
)

normal_path = dataset_path / "Normal"
outer_race_path = dataset_path / "Outer_Race"

print("Dataset exists:", dataset_path.exists())
print("Normal folder exists:", normal_path.exists())
print("Outer Race folder exists:", outer_race_path.exists())

Dataset exists: True
Normal folder exists: True
Outer Race folder exists: True


In [5]:
normal_files = sorted(normal_path.glob("*.mat"))
outer_files = sorted(outer_race_path.glob("*.mat"))

print("NORMAL FILES:")
for file in normal_files:
    print(file.name)

print("\nOUTER RACE FILES:")
for file in outer_files:
    print(file.name)

print("\nTotal Normal files:", len(normal_files))
print("Total Outer Race files:", len(outer_files))
print("Total recordings:", len(normal_files) + len(outer_files))

NORMAL FILES:
Normal_0.mat
Normal_1.mat
Normal_2.mat
Normal_3.mat

OUTER RACE FILES:
0R007_12_0HP.mat
0R007_12_1HP.mat
0R007_3_0HP.mat
0R007_3_1HP.mat
0R007_6_0HP.mat
0R007_6_1HP.mat
0R021_12_0HP.mat
0R021_6_0HP.mat
OR021_3_0HP.mat

Total Normal files: 4
Total Outer Race files: 9
Total recordings: 13


In [9]:
#Cell 4 – Set signal processing parameters
sampling_rate = 12000

window_size = 2048
overlap = 1024
step_size = window_size - overlap

print("Sampling rate:", sampling_rate)
print("Window size:", window_size)
print("Overlap:", overlap)
print("Step size:", step_size)
print("Window duration:", window_size / sampling_rate, "seconds")

Sampling rate: 12000
Window size: 2048
Overlap: 1024
Step size: 1024
Window duration: 0.17066666666666666 seconds


In [11]:
#Cell 5 – Create the feature extraction function
def extract_features(signal):

    rms = np.sqrt(np.mean(signal ** 2))

    features = {
        "Mean": np.mean(signal),
        "RMS": rms,
        "Standard_Deviation": np.std(signal),
        "Variance": np.var(signal),
        "Minimum": np.min(signal),
        "Maximum": np.max(signal),
        "Peak_to_Peak": np.ptp(signal),
        "Kurtosis": kurtosis(signal),
        "Skewness": skew(signal),
        "Crest_Factor": np.max(np.abs(signal)) / (rms + 1e-10)
    }

    return features

In [13]:
#Cell 6 – Create the windowing function
def create_windows(signal):

    windows = []

    for start in range(
        0,
        len(signal) - window_size + 1,
        step_size
    ):

        end = start + window_size

        window = signal[start:end]

        windows.append(window)

    return np.array(windows)

In [47]:
# Cell 7 – Load all recordings and extract features

feature_rows = []

all_files = (
    [(file, "Normal") for file in normal_files] +
    [(file, "Outer_Race") for file in outer_files]
)

for file_path, label in all_files:

    print("Processing:", file_path.name)

    data = loadmat(file_path)

    # Find all Drive-End signals
    de_keys = [
        key for key in data.keys()
        if key.endswith("_DE_time")
    ]

    if len(de_keys) == 0:
        print("No Drive-End signal found:", file_path.name)
        continue

    # Process every Drive-End signal separately
    for signal_id, key in enumerate(de_keys):

        signal = data[key].squeeze()

        print("  Signal:", key)
        print("  Samples:", len(signal))

        windows = create_windows(signal)

        for window_id, window in enumerate(windows):

            features = extract_features(window)

            features["Label"] = label
            features["Window_ID"] = window_id
            features["Source_File"] = file_path.name
            features["Signal_ID"] = key

            feature_rows.append(features)

complete_df = pd.DataFrame(feature_rows)

print("\nDataset processing completed!")
print("Dataset shape:", complete_df.shape)

Processing: Normal_0.mat
  Signal: X097_DE_time
  Samples: 243938
Processing: Normal_1.mat
  Signal: X098_DE_time
  Samples: 483903
Processing: Normal_2.mat
  Signal: X098_DE_time
  Samples: 483903
  Signal: X099_DE_time
  Samples: 485063
Processing: Normal_3.mat
  Signal: X100_DE_time
  Samples: 485643
Processing: 0R007_12_0HP.mat
  Signal: X156_DE_time
  Samples: 122281
Processing: 0R007_12_1HP.mat
  Signal: X158_DE_time
  Samples: 121991
Processing: 0R007_3_0HP.mat
  Signal: X144_DE_time
  Samples: 122281
Processing: 0R007_3_1HP.mat
  Signal: X145_DE_time
  Samples: 121846
Processing: 0R007_6_0HP.mat
  Signal: X130_DE_time
  Samples: 121991
Processing: 0R007_6_1HP.mat
  Signal: X131_DE_time
  Samples: 122426
Processing: 0R021_12_0HP.mat
  Signal: X258_DE_time
  Samples: 121846
Processing: 0R021_6_0HP.mat
  Signal: X234_DE_time
  Samples: 122426
Processing: OR021_3_0HP.mat
  Signal: X246_DE_time
  Samples: 121701

Dataset processing completed!
Dataset shape: (3183, 14)


In [49]:
# Cell 8 – Create the complete DataFrame
complete_df = pd.DataFrame(feature_rows)

print("Complete dataset shape:", complete_df.shape)

print("\nFirst five rows:")
display(complete_df.head())

Complete dataset shape: (3183, 14)

First five rows:


,Mean,RMS,Standard_Deviation,Variance,Minimum,Maximum,Peak_to_Peak,Kurtosis,Skewness,Crest_Factor,Label,Window_ID,Source_File,Signal_ID
0,0.012005,0.073256,0.072266,0.005222,-0.272869,0.217794,0.490663,-0.045824,-0.137322,3.724845,Normal,0,Normal_0.mat,X097_DE_time
1,0.013116,0.074507,0.073344,0.005379,-0.272869,0.212370,0.485239,-0.116338,-0.010099,3.662321,Normal,1,Normal_0.mat,X097_DE_time
2,0.012790,0.079035,0.077993,0.006083,-0.226765,0.223010,0.449775,-0.184580,-0.062348,2.869171,Normal,2,Normal_0.mat,X097_DE_time
3,0.012148,0.077916,0.076964,0.005923,-0.226765,0.223010,0.449775,-0.150657,-0.103819,2.910361,Normal,3,Normal_0.mat,X097_DE_time
4,0.011959,0.074338,0.073370,0.005383,-0.202148,0.219255,0.421403,-0.221760,0.025960,2.949440,Normal,4,Normal_0.mat,X097_DE_time


In [51]:
print("CLASS DISTRIBUTION:")
print(complete_df["Label"].value_counts())

print("\nRECORDING DISTRIBUTION:")
print(complete_df.groupby(
    ["Source_File", "Label"]
).size())

print("\nMISSING VALUES:")
print(complete_df.isnull().sum())

print("\nDATASET INFORMATION:")
complete_df.info()

CLASS DISTRIBUTION:
Label
Normal        2124
Outer_Race    1059
Name: count, dtype: int64

RECORDING DISTRIBUTION:
Source_File       Label     
0R007_12_0HP.mat  Outer_Race    118
0R007_12_1HP.mat  Outer_Race    118
0R007_3_0HP.mat   Outer_Race    118
0R007_3_1HP.mat   Outer_Race    117
0R007_6_0HP.mat   Outer_Race    118
0R007_6_1HP.mat   Outer_Race    118
0R021_12_0HP.mat  Outer_Race    117
0R021_6_0HP.mat   Outer_Race    118
Normal_0.mat      Normal        237
Normal_1.mat      Normal        471
Normal_2.mat      Normal        943
Normal_3.mat      Normal        473
OR021_3_0HP.mat   Outer_Race    117
dtype: int64

MISSING VALUES:
Mean                  0
RMS                   0
Standard_Deviation    0
Variance              0
Minimum               0
Maximum               0
Peak_to_Peak          0
Kurtosis              0
Skewness              0
Crest_Factor          0
Label                 0
Window_ID             0
Source_File           0
Signal_ID             0
dtype: int64

DATASET 

In [53]:
results_path = Path(
    r"C:\Users\suthi\Downloads\AI_Motor_Predictive_Maintenance\results"
)

results_path.mkdir(parents=True, exist_ok=True)

output_file = results_path / "complete_bearing_features.csv"

complete_df.to_csv(output_file, index=False)

print("Dataset saved successfully!")
print(output_file)

Dataset saved successfully!
C:\Users\suthi\Downloads\AI_Motor_Predictive_Maintenance\results\complete_bearing_features.csv


In [55]:
print("Dataset shape:", complete_df.shape)

print("\nClass distribution:")
print(complete_df["Label"].value_counts())

print("\nNumber of original recordings:")
print(complete_df["Source_File"].nunique())

print("\nMissing values:")
print(complete_df.isnull().sum().sum())

Dataset shape: (3183, 14)

Class distribution:
Label
Normal        2124
Outer_Race    1059
Name: count, dtype: int64

Number of original recordings:
13

Missing values:
0


In [25]:
# Check all downloaded recordings

all_files = (
    list(normal_path.glob("*.mat")) +
    list(outer_race_path.glob("*.mat"))
)

processed_files = set(complete_df["Source_File"].unique())

print("EXPECTED FILES:", len(all_files))
print("PROCESSED FILES:", len(processed_files))

print("\nFILES NOT PROCESSED:")

for file in all_files:
    if file.name not in processed_files:
        print(file.name)

print("\nPROCESSED RECORDINGS:")
for file in sorted(processed_files):
    print(file)

EXPECTED FILES: 13
PROCESSED FILES: 12

FILES NOT PROCESSED:
Normal_2.mat

PROCESSED RECORDINGS:
0R007_12_0HP.mat
0R007_12_1HP.mat
0R007_3_0HP.mat
0R007_3_1HP.mat
0R007_6_0HP.mat
0R007_6_1HP.mat
0R021_12_0HP.mat
0R021_6_0HP.mat
Normal_0.mat
Normal_1.mat
Normal_3.mat
OR021_3_0HP.mat


In [31]:
missing_file = "PUT_MISSING_FILENAME_HERE"

file_path = None

for folder in [normal_path, outer_race_path]:
    candidate = folder / missing_file
    if candidate.exists():
        file_path = candidate
        break

if file_path is not None:
    test_data = loadmat(file_path)
    print(test_data.keys())

In [33]:
missing_file = normal_path / "Normal_2.mat"

test_data = loadmat(missing_file)

print("MATLAB VARIABLES:")
print(test_data.keys())

MATLAB VARIABLES:
dict_keys(['__header__', '__version__', '__globals__', 'ans', 'X098_DE_time', 'X098_FE_time', 'X099_DE_time', 'X099_FE_time'])


In [35]:
de_keys = [
    key for key in test_data.keys()
    if key.endswith("_DE_time")
]

print("Drive-End variables:", de_keys)

Drive-End variables: ['X098_DE_time', 'X099_DE_time']


In [37]:
for key in de_keys:
    signal = test_data[key].squeeze()

    print("\nVariable:", key)
    print("Shape:", signal.shape)
    print("Total samples:", len(signal))
    print("First 10 values:", signal[:10])


Variable: X098_DE_time
Shape: (483903,)
Total samples: 483903
First 10 values: [ 0.046104   -0.03713354 -0.089496   -0.08490646 -0.03859385  0.02545108
  0.04068     0.04255754  0.03817662  0.04401785]

Variable: X099_DE_time
Shape: (485063,)
Total samples: 485063
First 10 values: [ 0.06425354  0.06300185 -0.00438092 -0.03588185 -0.02399077  0.00521538
  0.03024923  0.00751015 -0.01648062 -0.04193169]


In [39]:
rpm_keys = [key for key in test_data.keys() if "RPM" in key]

print("RPM variables:", rpm_keys)

for key in rpm_keys:
    print(key, test_data[key].squeeze())

RPM variables: []


In [45]:
print("MAT-file header:")
print(test_data["__header__"].decode(errors="ignore"))

print("\nAll variables in the file:")

for key, value in test_data.items():
    if not key.startswith("__"):
        print(
            "Variable:", key,
            "| Shape:", value.shape,
            "| Data type:", value.dtype
        )

for key in test_data.keys():
    if not key.startswith("__"):
        print(key)

MAT-file header:
MATLAB 5.0 MAT-file, Platform: PCWIN, Created on: Fri Jan 28 11:19:25 2000

All variables in the file:
Variable: ans | Shape: (1, 1) | Data type: uint8
Variable: X098_DE_time | Shape: (483903, 1) | Data type: float64
Variable: X098_FE_time | Shape: (483903, 1) | Data type: float64
Variable: X099_DE_time | Shape: (485063, 1) | Data type: float64
Variable: X099_FE_time | Shape: (485063, 1) | Data type: float64
ans
X098_DE_time
X098_FE_time
X099_DE_time
X099_FE_time
